In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_7.pickle

In [ ]:
%%cudf.pandas.profile
### cell 8 ###

# cell 8 optimized
# Compute first and last values per group entirely on GPU
cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

# Mobile stats
g = Mobile_df.groupby('Name')
first_m = g[cols].first()
last_m  = g[cols].last()
Mobile_Stats = (
    (last_m['Avg. Avg D Kbps'] - first_m['Avg. Avg D Kbps'])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(last_m['Avg. Avg U Kbps'] - first_m['Avg. Avg U Kbps']),
        Change_Latency=(last_m['Avg Lat Ms']      - first_m['Avg Lat Ms'])
    )
)

# Broadband stats
g = Broadband_df.groupby('Name')
first_b = g[cols].first()
last_b  = g[cols].last()
Broadband_Stats = (
    (last_b['Avg. Avg D Kbps'] - first_b['Avg. Avg D Kbps'])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(last_b['Avg. Avg U Kbps'] - first_b['Avg. Avg U Kbps']),
        Change_Latency=(last_b['Avg Lat Ms']      - first_b['Avg Lat Ms'])
    )
)

# Combine the two change‐in‐download series with a GPU join
Total_Stats = (
    Mobile_Stats['Change_Download']
    .to_frame('Mobile')
    .join(
        Broadband_Stats['Change_Download']
        .to_frame('Broadband')
    )
)
